In [0]:

#Leyendo la tabla de caracteristicas generales
ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")

# Convertir el DataFrame de Spark a Pandas
ventas = ventas.toPandas()

ventas

In [0]:
import pandas as pd
 
# tipos de datos
print(ventas.dtypes)
 
# nulos por columna
nulos_total = ventas.isnull().sum()
nulos_pct = (ventas.isnull().mean() * 100).round(2)
print(pd.DataFrame({'total_nulos': nulos_total, 'pct': nulos_pct}))
 
# descriptivas, solo las numéricas que importan
print(ventas[['cantidad', 'precio_unitario']].describe())

Fase 2 - Limpieza

In [0]:
# quito las transacciones repetidas
ventas = ventas.drop_duplicates(subset=['id_transaccion'])
 
# 'Tarjeta Credito' venía escrito de dos formas
ventas['metodo_pago'] = ventas['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})
 
# nulos: mediana en cantidad, moda en satisfacción
mediana = ventas['cantidad'].median()
ventas['cantidad'] = ventas['cantidad'].fillna(mediana)
 
moda = ventas['satisfaccion_cliente'].mode()[0]
ventas['satisfaccion_cliente'] = ventas['satisfaccion_cliente'].fillna(moda)
 
# precios de 9999, outlier claro
filtro_outlier = ventas[ventas['precio_unitario'] == 9999].index
ventas = ventas.drop(filtro_outlier)



Fase 3 - Transformación e Ingeniería de Atributos

In [0]:
import pandas as pd

ventas['monto_total'] = ventas['cantidad'] * ventas['precio_unitario']

ventas['fecha'] = pd.to_datetime(ventas['fecha'])
ventas['dia_semana'] = ventas['fecha'].dt.day_name()
ventas['mes'] = ventas['fecha'].dt.month

ventas['rango_monto'] = pd.cut(ventas['monto_total'], bins=[0, 100, 500, 999999], labels=['Bajo', 'Medio', 'Alto'])

Fase 4: Agregación y Resumen de Datos

In [0]:
# monto total y cantidad promedio por categoría
resumen_productos = ventas.groupby('categoria_producto').agg({'monto_total': 'sum', 'cantidad': 'mean'})
 
# región vs categoría
matriz = pd.pivot_table(ventas, values='monto_total', index='region', columns='categoria_producto')
 
# satisfacción promedio por método de pago
satisfaccion = ventas.groupby('metodo_pago')['satisfaccion_cliente'].mean()

Fase 5: Selección, Filtrado y Muestreo

In [0]:
# Norte con satisfacción de 4 o más
ventas_norte_exitosas = ventas[(ventas['region'] == 'Norte') & (ventas['satisfaccion_cliente'] >= 4)]
 
# 10% de cada región para no perder las proporciones
muestra = ventas.groupby('region').sample(frac=0.10)